# Phase 0: Build Schema Translation Dictionary

This notebook builds the schema translation dictionary for IDSpider1.
It extracts all identifiers from Spider's `tables.json`, translates
individual words via Google Translate, and builds a collision-free
identifier mapping.

**Output**: `schema_translation_dict.json` — download and use in Phase 1.

**Estimated API requests**: ~1500-2000 unique words

In [ ]:
!pip install -q googletrans==4.0.2 tqdm tenacity nest_asyncio

In [ ]:
import os, sys, zipfile, urllib.request

# === CONFIGURATION ===
SPIDER_ZIP_URL = 'https://github.com/mfazrinizar/mfazrinizar/releases/download/1.0.0/spider.zip'
WORK_DIR = '/kaggle/working'
SPIDER_DIR = os.path.join(WORK_DIR, 'spider')
OUTPUT_DIR = os.path.join(WORK_DIR, 'phase0')
SRC_DIR = '/kaggle/input/idspider1-src-new'  # Upload src_new/ as Kaggle dataset

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Download and extract Spider dataset
zip_path = os.path.join(WORK_DIR, 'spider.zip')
if not os.path.exists(SPIDER_DIR):
    print('Downloading spider.zip...')
    urllib.request.urlretrieve(SPIDER_ZIP_URL, zip_path)
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORK_DIR)
    os.remove(zip_path)
    print(f'Spider dataset at: {SPIDER_DIR}')
else:
    print('Spider dataset already exists')

print('Contents:', os.listdir(SPIDER_DIR))

In [ ]:
# Add src_new to path
sys.path.insert(0, os.path.dirname(SRC_DIR))

from pathlib import Path
from src_new.schema_translator import run_phase0

tables_json = Path(SPIDER_DIR) / 'tables.json'
dict_path = Path(OUTPUT_DIR) / 'schema_translation_dict.json'
words_path = Path(OUTPUT_DIR) / 'schema_translation_dict_words.json'
collision_path = Path(OUTPUT_DIR) / 'collision_report.json'

trans_dict = run_phase0(
    tables_json_path=tables_json,
    output_dict_path=dict_path,
    output_words_path=words_path,
    collision_report_path=collision_path,
    use_sync=True,
)

print(f'\nDictionary entries: {len(trans_dict)}')
print(f'Download {dict_path} and upload as Kaggle dataset for Phase 1.')

In [ ]:
# Show sample translations
import json
changed = {k: v for k, v in trans_dict.items() if k != v}
print(f'Changed: {len(changed)}/{len(trans_dict)}')
for k, v in list(changed.items())[:30]:
    print(f'  {k:30s} -> {v}')

### Next Steps

1. Download `schema_translation_dict.json` from `/kaggle/working/phase0/`
2. Upload as a Kaggle dataset (e.g., `idspider1-schema-dict`)
3. Run Phase 1 notebook to translate questions